# 03 - Hybrid BM25 + dense retrieval (RRF fusion)

BM25 catches exact-term hits, the dense bi-encoder catches paraphrases.
Reciprocal Rank Fusion combines the two ranked lists in a way that's robust
to the score-scale mismatch (BM25 is unbounded; cosine sits in `[-1, 1]`).

In [1]:
import os
import sys
import warnings

sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv

from rag.data_ingestion import chunk_documents, load_documents
from rag.llm import LLM
from rag.metrics import embedding_faithfulness
from rag.pipeline import RAGPipeline
from rag.retrievers import HybridRetriever

warnings.filterwarnings('ignore')
load_dotenv(os.path.abspath('../.env'))

FILE_PATH = '../data/google_10K.pdf'
QUERY = 'Provide operating income, operating margin, and net income for every fiscal year reported in the income statement.'

docs = load_documents(FILE_PATH)
chunks = chunk_documents(docs, chunk_size=2000, chunk_overlap=200)
print(f'Loaded {len(docs)} pages -> {len(chunks)} chunks')

Loaded 107 pages -> 230 chunks


In [2]:
retriever = HybridRetriever(rank_constant=60, candidates_per_retriever=20)
retriever.add_documents(chunks)

for i, hit in enumerate(retriever.retrieve(QUERY, k=3), start=1):
    snippet = hit.document.page_content[:200].replace('\n', ' ')
    print(f'{i}. (RRF={hit.score:.4f}) {snippet}...')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1. (RRF=0.0325) Table of Contents Alphabet Inc. • employee compensation expenses for employees in finance, human resources, information technology, legal, and other administrative support functions; • expenses relati...
2. (RRF=0.0305) Table of Contents Alphabet Inc. Year Ended December 31,  AOCI Components Location 2023 2024 2025 Unrealized gains (losses) on available-for-sale investments Other income (expense), net $ (1,497) $ (1,...
3. (RRF=0.0299) Table of Contents Alphabet Inc. Alphabet Inc. CONSOLIDATED STATEMENTS OF COMPREHENSIVE INCOME (in millions)   Year Ended December 31,   2023 2024 2025 Net income $ 73,795  $ 100,118  $ 132,170  Other ...


In [3]:
llm = LLM(api_key=os.environ['GROQ_API_KEY']).get_llm(
    provider='groq',
    model_name='llama-3.3-70b-versatile',
    temperature=0.0,
)
pipeline = RAGPipeline(retriever=retriever, llm=llm, top_k=8)
response = pipeline.answer(QUERY)
print(response.answer)

Based on the provided context, the operating income, operating margin, and net income for the reported fiscal years are as follows:

- For the year ended December 31, 2024: 
  - Operating income: $112,390 million (Doc 4, page 42)
  - Operating margin: 32% (Doc 7, page 38)
  - Net income: $100,118 million (Doc 1, page 38 and Doc 7, page 38)

- For the year ended December 31, 2025: 
  - Operating income: $129,039 million (Doc 4, page 42)
  - Operating margin: 32% (Doc 7, page 38)
  - Net income: $132,170 million (Doc 1, page 38 and Doc 7, page 38)

- For the year ended December 31, 2023: 
  - Operating income: Not directly provided in the context.
  - Operating margin: Not directly provided in the context.
  - Net income: $73,795 million (Doc 3, page 56)


## Inline evaluation

`embedding_faithfulness` checks the answer is semantically close to the
retrieved context.

In [4]:
context_strings = [doc.page_content for doc in response.contexts]
faith = embedding_faithfulness(response.answer, context_strings)
print(f'embedding_faithfulness = {faith:.3f}')

embedding_faithfulness = 0.576
